# Optimus 5-Prime for 5′ UTR design

[Notebook](single-task.ipynb) · [Installation](../docs/installation.md)

Maximize predicted mean ribosome load for a 50-nt 5′ UTR. Exclude homopolymers of six or more bases.

## 1. Load the model



In [1]:
# %pip install "bioseq-models @ git+https://github.com/UTR-CODE/bioseq-models.git@84a5075c3656127d99027c38705212a2de6e62c9"

import re
from pathlib import Path

import torch
from bioseq_models import load_model
from MotifSift import call_motifs, generate_designs, random_seed_sequences, select_candidates

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
model = load_model("optimus5prime", device=device)

run = Path("outputs/optimus_5utr")




## 2. Generate candidates



In [2]:
def no_homopolymer6(sequences):
    return [re.search(r"([ACGT])\1{5}", sequence) is None for sequence in sequences]


seeds = random_seed_sequences(50, 100, seed=42)

designs = generate_designs(
    model, length=50, initial=seeds, objective="maximize",
    constraint=no_homopolymer6,
    output_dir=run / "designs",
)
designs.groupby("algorithm")["score"].agg(["size", "median", "max"])


,size,median,max
algorithm,,,
adalead,100,9.267762,9.572426
basesearch,100,8.857733,9.130070
beam_search,100,9.321070,9.762297
cross_entropy_method,100,9.378455,9.511713
den,100,9.065636,9.345160
fast_seqprop,100,8.949476,9.323884
genetic_algorithm,100,9.488063,9.610309
ism_greedy,100,9.328421,9.639274
ledidi,100,8.767134,9.056448




## 3. Discover motifs



In [3]:
motifs = call_motifs(designs, model, reverse_complement=False, output_dir=run / "motifs")
motifs.support.query("retained")[["motif_id", "seqlets", "parents", "sites_on_unique_DNA"]]


,motif_id,seqlets,parents,sites_on_unique_DNA
0,1-UAUAAUUK,21,21,609.0
1,2-AUAWAKUGU,28,28,381.0
2,3-AGGAAUUG,14,14,44.0
3,4-UUAUAUUG,17,17,1125.0
12,18-AUAUUAUAC,12,12,390.0
18,5-AUAUAGUAA,17,17,378.0




## 4. Select a panel



In [4]:
gate = float(designs.drop_duplicates("sequence").score.quantile(0.2))
selected = select_candidates(designs, motifs=motifs, top_n=10, min_score=gate, output_dir=run / "selected")
selected[["selection_rank", "sequence", "score", "present_motifs"]]


,selection_rank,sequence,score,present_motifs
0,1,TTGATTTTAATTTAATTGATATAGATTTTATTGTTTTTAAATTAAT...,9.762297,1-UAUAAUUK;2-AUAWAKUGU;4-UUAUAUUG
517,2,GAATATTTGTTTAAACTTTGTTCTTGACAAAAGAATTAATTAGTGA...,9.160056,3-AGGAAUUG
72,3,TTGATTTTAGTCTAATTCTTTTATTCTTTTACTATTTAGTTAAAAT...,9.502949,5-AUAUAGUAA
835,4,TTGATATTTACAAAGTCGAAACTAATAAAAGTAGAAAAGAATTTTA...,8.818485,3-AGGAAUUG;18-AUAUUAUAC
760,5,TAATATTTTTATAGAAATTGATTCTAAAAAGAAAGTTAATAATAGA...,8.866099,2-AUAWAKUGU;3-AGGAAUUG;4-UUAUAUUG;5-AUAUAGUAA
205,6,TTTTTGTTATATTATTATTAAATTTTATTTTAATTAAAATTAAATA...,9.406013,4-UUAUAUUG;18-AUAUUAUAC
5,7,GAGAGAGAGATTTGATTTTTGTTATTGTTTTAATTTTTAAAATTAG...,9.658700,1-UAUAAUUK
374,8,GTGAGATTTGTAGCATAACCTGAAGAATTTAATTTTAATTTAATAA...,9.278208,1-UAUAAUUK;3-AGGAAUUG;5-AUAUAGUAA
443,9,ACCCAAATATAGTTTTATACTTCTTGATTTCTGAATTTAGAATTAA...,9.234110,2-AUAWAKUGU;18-AUAUUAUAC;5-AUAUAGUAA
654,10,TCAGAGTCTTCTTAATTATAATAGACCCGTCTAAAAACTTTGAAAC...,8.967864,1-UAUAAUUK;4-UUAUAUUG;18-AUAUUAUAC;5-AUAUAGUAA
